In [ ]:
from pathlib import Path
import sys

if repo_root.name == 'scripts':
    repo_root = repo_root.parent
if not (repo_root / 'scripts').exists():
    raise RuntimeError(f'Could not locate repo root from {repo_root}')

sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
mne.set_log_level('WARNING')

from scripts import config
from scripts.decoding.epoch_io import get_epochs_path, load_epochs
from scripts.decoding.decoding_utils import (
    run_group_decoding, 
    save_outputs,
)
output_dir = repo_root / 'output_mne' / 'decoding'

In [ ]:
PIPELINE = 'proposed'
SUBJECTS = sorted(config.SUBJECT_INFO.keys())
CONTEXTS = ['mid_high', 'high_high']
WINDOW_START = 0.20
WINDOW_END = 0.35

# dchecking the first 5 subjects
SUBJECTS[:5]

In [ ]:
example_subject = SUBJECTS[0]
epochs_path = get_epochs_path(example_subject, PIPELINE, 'feedback')
epochs = load_epochs(example_subject, PIPELINE, 'feedback', preload=False)

print(epochs_path)
print('n_epochs:', len(epochs))
print('metadata columns:', list(epochs.metadata.columns))
display(
    epochs.metadata[['context', 'cue_value', 'outcome_label']]
    .value_counts()
    .rename('n')
    .reset_index()
    .head(10)
)

In [ ]:
summary_df, group_stats, timecourse_store = run_group_decoding( 
    SUBJECTS, 
    PIPELINE, 
    CONTEXTS, 
    window_start=WINDOW_START, 
    window_end=WINDOW_END, 
) 

summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for context in CONTEXTS:
    curves = [timecourse_store[sid][context]['mean_scores'] for sid in SUBJECTS if sid in timecourse_store and context in timecourse_store[sid]]
    times = next(timecourse_store[sid][context]['times'] for sid in SUBJECTS if sid in timecourse_store and context in timecourse_store[sid])
    mean_curve = np.mean(curves, axis=0)
    ax.plot(times, mean_curve, label=context)
    ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
    
ax.axvspan(WINDOW_START, WINDOW_END, color='grey', alpha=0.15)
ax.set(xlabel='Time (s)', ylabel='AUC', title='Outcome decoding by context')
ax.legend()
plt.show()

In [ ]:
summary_path, stats_path, npz_path = save_outputs(output_dir, PIPELINE, CONTEXTS, summary_df, group_stats, timecourse_store)
print(summary_path)
print(stats_path)
print(npz_path)
group_stats